# Урок 2.5. Чанкинг, метаданные и reranking

**Модуль 2 «Embeddings и векторные базы данных» · Курс «LLM для Python-разработчиков»**

До сих пор мы индексировали документы **целиком** — и это работало, потому что база маленькая и опрятная. Сегодня разберём три практики, без которых семантический поиск не выживает на реальных данных: **чанкинг** (как резать тексты), **метаданные чанков** (как не потерять связь фрагмента с источником) и **reranking** (как пересортировать выдачу перед показом). Как всегда — не на веру, а с замерами.

## 🎯 Цели урока

После этого урока вы сможете:

- объяснить, почему вектор целого документа — «среднее по больнице», и что такое **разрешение поиска**;
- применять две базовые стратегии чанкинга — **фиксированное окно с перекрытием** и **по абзацам** — и знать их компромиссы;
- проектировать **метаданные чанка**: паспорт фрагмента, без которого он — сирота;
- строить **двухступенчатый поиск** (быстрый recall → точный precision) и простейший reranking-диверсификацию;
- выбирать размер чанка **eval'ом**, а не интуицией.

## План

1. Проблема: разрешение поиска
2. Стратегия 1: окно с перекрытием
3. Стратегия 2: по абзацам
4. Метаданные чанка
5. Меряем: документы против чанков
6. Reranking: вторая ступень
7. То же с эмбеддингами (для владельцев модели)

⏱ **Время:** ~75–90 минут.

> 💡 Почти весь урок живой без скачиваний: механика чанкинга — чистый Python, а для честных сравнений мы соберём крошечный лексический поиск. Модель e5 понадобится только в секции 7.

## 1. Проблема: разрешение поиска

Откройте глазами документ «Больничные» из нашей базы: правило трёх дней доверия, оформление ЭЛН, доплата до 100% оклада, запрет работать во время болезни. **Четыре разные подтемы.** Один вектор на весь документ — это *среднее* всех подтем: точка на карте смыслов, которая не совпадает ни с одной из них.

```text
              карта смыслов (условно)

      ² правило 3 дней        ² оформление ЭЛН
              \                 /
               * <- вектор ВСЕГО документа
              /                 \
      ² доплата 100%          ² запрет работать
      ^
      сюда попадает вопрос "сколько доплачивают?"
      ближайшая точка - подтема, а не среднее!
```

Вопрос пользователя всегда бьёт в **подтему**. Чем длиннее документ, тем дальше его «средний» вектор от любого конкретного вопроса. Отсюда два вывода-крайности:

- индексировать **целые документы** — контекст сохранён, но вектор размыт;
- индексировать **отдельные предложения** — векторы точные, но фрагмент без контекста бесполезен («Компания доплачивает до 100%» — чего? кому? когда?).

**Чанк** (chunk) — компромисс между этими крайностями, и его размер — это буквально *разрешение* вашего поиска, как размер пикселя. Посмотрим, с чем имеем дело:

In [ ]:
from utils import chunk_fixed, chunk_paragraphs, load_documents

docs = load_documents("data")

print(f"{'документ':<15} {'символов':>9} {'абзацев':>8}")
print("-" * 36)
for doc in docs:
    n_paragraphs = len([p for p in doc.text.split("\n\n") if p.strip()])
    print(f"{doc.doc_id:<15} {len(doc.text):>9} {n_paragraphs:>8}")

Документы по 600–900 символов и 4–6 абзацев — маленькие, но уже многотемные. Реальные вики-страницы и регламенты в 10–50 раз длиннее, и всё, что мы сейчас увидим, там усиливается пропорционально.

## 2. Стратегия 1: окно с перекрытием

Самый простой чанкер — скользящее окно фиксированного размера. В [`utils.py`](utils.py) он уже написан — `chunk_fixed(text, chunk_size, overlap)`, с двумя важными деталями:

- окно **не режет слова**: правая граница откатывается к ближайшему пробелу;
- соседние чанки **перекрываются** на `overlap` символов.

Зачем перекрытие? Мысль, которой не повезло оказаться на границе, будет разрезана пополам — и не найдётся ни по одному запросу. Перекрытие даёт каждой границе «вторую попытку»: хвост чанка N повторяется в голове чанка N+1.

In [ ]:
bolnichny = next(d for d in docs if d.doc_id == "bolnichny")

chunks = chunk_fixed(bolnichny.text, chunk_size=300, overlap=60)
print(f"Документ {len(bolnichny.text)} символов -> {len(chunks)} чанков по ~300\n")
for i, chunk in enumerate(chunks):
    print(f"--- чанк {i} ({len(chunk)} симв.): {chunk[:60]}...")

# перекрытие можно увидеть: конец чанка 0 совпадает с началом чанка 1
tail = chunks[0][-40:]
print(f"\nхвост чанка 0:  ...{tail!r}")
print(f"голова чанка 1: {chunks[1][:40]!r}...")

print(f"\n{'chunk_size':>10} {'чанков':>7} {'средняя длина':>14}")
for size in (200, 400, 800):
    pieces = chunk_fixed(bolnichny.text, chunk_size=size, overlap=size // 5)
    avg = sum(map(len, pieces)) / len(pieces)
    print(f"{size:>10} {len(pieces):>7} {avg:>14.0f}")

Как выбирать размер? Отправные точки (потом всё равно померяем):

- **300–600 символов** (~1–2 абзаца) — рабочая зона для факт-вопросов вроде наших;
- **перекрытие 15–25%** от размера — меньше почти не защищает, больше — раздувает индекс дубликатами;
- жёсткий потолок сверху: **лимит модели эмбеддингов**. У e5-small это 512 токенов (~1000–1500 символов русского текста — вспомните урок 1.1 про токены). Чанк длиннее — и модель **тихо обрежет хвост** при эмбеддинге: хвост просто не попадёт в вектор. Знакомый паттерн тихой обрезки из урока 1.5, только теперь на этапе индексации — самый незаметный вид потери данных.

## 3. Стратегия 2: по абзацам

Фиксированное окно слепо: ему всё равно, где автор поставил заголовок или закончил мысль. А ведь **абзац — это готовый чанк, размеченный человеком**: автор документа уже сгруппировал предложения по смыслу.

`chunk_paragraphs(text, max_chars)` из utils.py режет по пустым строкам, склеивает соседние короткие абзацы (чтобы не плодить микрочанки) и дорезает аномально длинные через `chunk_fixed`. Сравним обе стратегии на инструкции по VPN — документе с нумерованными шагами:

In [ ]:
vpn = next(d for d in docs if d.doc_id == "vpn")

print("=== chunk_fixed(300): границы не знают о структуре ===")
for i, chunk in enumerate(chunk_fixed(vpn.text, chunk_size=300, overlap=60)):
    first_line = chunk.replace("\n", " | ")[:70]
    print(f"  чанк {i}: {first_line}...")

print("\n=== chunk_paragraphs(500): границы = абзацы автора ===")
for i, chunk in enumerate(chunk_paragraphs(vpn.text, max_chars=500)):
    first_line = chunk.replace("\n", " | ")[:70]
    print(f"  чанк {i}: {first_line}...")

Присмотритесь к границам: фиксированное окно рвёт текст посреди нумерованных шагов, абзацный чанкер держит логические блоки целыми. Правило: **если у текста есть структура — используйте её.** Markdown-заголовки, абзацы, пункты списков — всё это бесплатная разметка смысловых границ. Фиксированное окно — запасной вариант для «текстовой каши» без структуры (и для дорезки слишком длинных абзацев, что `chunk_paragraphs` и делает).

## 4. Метаданные чанка: паспорт фрагмента

Разрезав документ, легко получить кучу безымянных фрагментов. Чанк без метаданных — сирота: его нельзя ни показать пользователю со ссылкой, ни отфильтровать, ни склеить обратно. Минимальный паспорт:

| поле | зачем |
|---|---|
| `doc_id` | связь с документом: дедупликация, «не более N чанков с документа» |
| `title` | что показать пользователю рядом с фрагментом |
| `topic` | фильтры из уроков 2.3–2.4 работают и на чанках |
| `chunk_index` | порядок: подтянуть соседей (упражнение 2), собрать документ обратно |
| `total_chunks` | понять, что фрагмент — «3 из 7», а не весь документ |
| `source` | путь к первоисточнику: аудит и доверие |

Соберём конвейер «документы → чанки с паспортами» — в utils.py для этого уже есть dataclass `Chunk`:

In [ ]:
from utils import Chunk

def chunk_documents(docs: list, max_chars: int = 500) -> list[Chunk]:
    """Конвейер индексации: каждый документ -> чанки с полными метаданными."""
    all_chunks: list[Chunk] = []
    for doc in docs:
        pieces = chunk_paragraphs(doc.text, max_chars=max_chars)
        for i, piece in enumerate(pieces):
            all_chunks.append(Chunk(
                chunk_id=f"{doc.doc_id}_{i}",
                doc_id=doc.doc_id,
                text=piece,
                metadata={
                    "title": doc.title,
                    "topic": doc.topic,
                    "chunk_index": i,
                    "total_chunks": len(pieces),
                    "source": doc.source,
                },
            ))
    return all_chunks

chunks = chunk_documents(docs)
print(f"{len(docs)} документов -> {len(chunks)} чанков\n")
sample = chunks[3]
print("Пример чанка:", sample.chunk_id)
print("  метаданные:", sample.metadata)
print("  текст:", sample.text[:120].replace("\n", " "), "...")

## 5. Меряем: документы против чанков

Пора проверить пользу чанкинга числами. Для этого нужен поиск, работающий прямо сейчас, без скачивания моделей, — соберём **лексический TF-IDF за 20 строк**. Идея из классического информационного поиска: вес слова = его частота в тексте (TF) × его редкость в корпусе (IDF). Слово «больничный» редкое и весит много, слово «в» есть везде и не весит ничего.

Это честный поиск со своей слабостью — он **не понимает синонимов** (урок 2.1: «пёс» ≠ «собака»), и мы эту слабость сейчас увидим во всей красе. Зато для сравнения «документы vs чанки» он идеален: одна и та же механика, меняется только гранулярность.

In [ ]:
import math
import re

def tokenize(text: str) -> list[str]:
    return re.findall(r"\w+", text.lower())

def build_lexical_index(texts: list[str]):
    """TF-IDF-лайт: списки токенов + словарь редкости слов (IDF)."""
    token_lists = [tokenize(t) for t in texts]
    df: dict[str, int] = {}
    for tokens in token_lists:
        for word in set(tokens):
            df[word] = df.get(word, 0) + 1
    idf = {word: math.log(len(texts) / count) for word, count in df.items()}
    return token_lists, idf

def lexical_search(query: str, token_lists, idf, top_k: int = 5):
    """Возвращает [(score, индекс_текста)] по убыванию релевантности."""
    scored = []
    for i, tokens in enumerate(token_lists):
        score = sum((tokens.count(word) / len(tokens)) * idf.get(word, 0.0)
                    for word in set(tokenize(query)))
        scored.append((score, i))
    return sorted(scored, reverse=True)[:top_k]

# два индекса: целые документы и чанки
doc_tokens, doc_idf = build_lexical_index([d.text for d in docs])
chunk_tokens, chunk_idf = build_lexical_index([c.text for c in chunks])

question = "Сколько компания доплачивает в первые дни болезни?"

print(f"Вопрос: {question}\n")
score, i = lexical_search(question, doc_tokens, doc_idf, top_k=1)[0]
print(f"ПО ДОКУМЕНТАМ: score={score:.3f}, нашли «{docs[i].title}» — {len(docs[i].text)} символов")

score, i = lexical_search(question, chunk_tokens, chunk_idf, top_k=1)[0]
best = chunks[i]
print(f"ПО ЧАНКАМ:     score={score:.3f}, нашли {best.chunk_id} — {len(best.text)} символов:")
print(f"\n>>> {best.text}")

Смотрите на два эффекта сразу:

1. **Score сконцентрировался** (у нас ~0.4 против ~0.17): в коротком чанке слова вопроса составляют заметную долю текста, в длинном документе они «разбавлены» тремя другими подтемами. То самое размытие, теперь в числах.
2. **Нашёлся ответ, а не место, где поискать**: конкретный абзац про доплату вместо документа на 740 символов. Для человека это удобство; для RAG (модуль 3) — критично: этот фрагмент поедет в промпт LLM, где каждый лишний символ — это деньги (урок 1.3) и шум, провоцирующий галлюцинации.

Теперь полный eval — наши вопросы из прошлых уроков плюс два новых «точечных», по обеим стратегиям:

In [ ]:
EVAL_QUESTIONS = [
    ("Сколько дней отпуска положено в году?",               "otpusk"),
    ("Как оформить больничный лист?",                       "bolnichny"),
    ("VPN подключён, но внутренние сайты не открываются",   "vpn"),
    ("Кто должен посмотреть мой код перед мержем?",         "code-review"),
    ("Упал прод, что делать в первую очередь?",             "incidents"),
    ("Сколько компания доплачивает в первые дни болезни?",  "bolnichny"),
    ("За сколько минут дежурный должен отреагировать ночью?", "incidents"),
]

def eval_strategy(label: str, texts: list[str], owners: list[str]) -> None:
    """owners[i] - doc_id владельца текста i (для чанков - родительский документ)."""
    token_lists, idf = build_lexical_index(texts)
    top1 = top3 = 0
    total_chars = 0
    marks = []
    for question, expected in EVAL_QUESTIONS:
        results = lexical_search(question, token_lists, idf, top_k=10)
        found_docs: list[str] = []
        for _, i in results:
            if owners[i] not in found_docs:
                found_docs.append(owners[i])
        top1 += found_docs[0] == expected
        top3 += expected in found_docs[:3]
        total_chars += len(texts[results[0][1]])
        marks.append("+" if found_docs[0] == expected else "x")
    print(f"{label:<22} top1={top1}/{len(EVAL_QUESTIONS)}  top3={top3}/{len(EVAL_QUESTIONS)}  "
          f"средний размер находки: {total_chars // len(EVAL_QUESTIONS)} симв.  {' '.join(marks)}")

print("вопросы:", " | ".join(q[:20] + "..." for q, _ in EVAL_QUESTIONS), "\n")
eval_strategy("целые документы", [d.text for d in docs], [d.doc_id for d in docs])
eval_strategy("чанки (по абзацам)", [c.text for c in chunks], [c.doc_id for c in chunks])

Два честных наблюдения по итогам замера:

1. **Точность «какой документ» не изменилась.** На 14 маленьких, чисто разделённых по темам документах чанкинг и не должен её менять — выигрыш в другом: находка стала в 2–3 раза короче и точнее (смотрите на средний размер). На реальных базах с длинными многотемными документами растёт и доля правильных документов — но проверяется это только eval'ом на своих данных.
2. **Оба уровня провалили «Упал прод»** — в документе про инциденты нет ни слова «упал», ни слова «прод». Это предел лексического поиска, и никакой чанкинг его не лечит. Лечат эмбеддинги (урок 2.1). Запомните этот вывод: **чанкинг и семантика — ортогональные улучшения**, они складываются, а не заменяют друг друга.

## 6. Reranking: вторая ступень

Поисковая выдача до сих пор формировалась одной формулой. Продакшен-поиск почти всегда **двухступенчатый**:

```text
этап 1: RECALL                      этап 2: PRECISION
дёшево и широко                     дорого и точно
--------------------------------    -------------------------------
вектор/BM25 по миллионам точек  ->  пересортировать топ-20-50
взять топ-20-50 кандидатов          умной моделью или правилами,
                                    отдать топ-3-5
```

Аналогия с наймом: этап 1 — скрининг резюме (быстрый фильтр по ключевым признакам), этап 2 — собеседование (дорогая, но точная оценка немногих). Реранжировать можно:

- **cross-encoder-моделью** — золотой стандарт, весь следующий урок про это;
- **LLM-судьёй** — гибко, дорого (вернёмся в модуле про RAG);
- **правилами и метаданными** — дёшево и сердито: свежесть, источник, разнообразие.

Последнее попробуем прямо сейчас. Типичная болезнь выдачи по чанкам: **весь топ забит фрагментами одного документа**. Пользователь спросил про две вещи — получил три куска одного текста. Лечение — диверсификация: не более N чанков с одного документа:

In [ ]:
def rerank_diverse(candidates, chunks, max_per_doc: int = 1, top_k: int = 3):
    """Этап 2: пересортировка с ограничением 'не более max_per_doc чанков с документа'."""
    picked, per_doc = [], {}
    for score, idx in candidates:
        owner = chunks[idx].doc_id
        if per_doc.get(owner, 0) >= max_per_doc:
            continue
        picked.append((score, idx))
        per_doc[owner] = per_doc.get(owner, 0) + 1
        if len(picked) == top_k:
            break
    return picked

question = "Как оформить больничный и отпуск?"
candidates = lexical_search(question, chunk_tokens, chunk_idf, top_k=10)

print(f"Вопрос: {question}\n")
print("ЭТАП 1 (топ-5 по score):")
for score, i in candidates[:5]:
    print(f"  {score:.3f}  {chunks[i].chunk_id}")

print("\nЭТАП 2 (диверсификация, максимум 1 чанк на документ):")
for score, i in rerank_diverse(candidates, chunks, max_per_doc=2):
    print(f"  {score:.3f}  {chunks[i].chunk_id:<14} {chunks[i].metadata['title']}")

Вопрос был про **две** темы — после диверсификации в топе представлены обе, а не три куска больничного. Заметьте, что этап 2 работает только с кандидатами этапа 1: **реранкер не может вернуть то, что recall-этап не принёс**. Отсюда правило баланса: кандидатов на входе второй ступени должно быть с запасом (20–50), но не тысячи — иначе теряется смысл дешёвого первого этапа.

(А в упражнении 3 вы убедитесь, что и диверсификация — не бесплатный обед.)

## 7. То же с эмбеддингами

Если модель e5 из урока 2.1 уже в кэше — прогоните семантическую версию сравнения. Логика та же, меняется только скоринг:

In [ ]:
import numpy as np

EMB_MODEL_NAME = "intfloat/multilingual-e5-small"

model = None
try:
    from sentence_transformers import SentenceTransformer
    from huggingface_hub import snapshot_download
    snapshot_download(EMB_MODEL_NAME, local_files_only=True)
    model = SentenceTransformer(EMB_MODEL_NAME)
    print("Модель загружена.")
except Exception:
    print("Модель e5 не в кэше (урок 2.1) — семантическое сравнение пропустится.")

if model is not None:
    def embed(texts: list[str], prefix: str) -> np.ndarray:
        return model.encode([f"{prefix}: {t}" for t in texts],
                            normalize_embeddings=True)

    doc_vecs = embed([d.text for d in docs], "passage")
    chunk_vecs = embed([c.text for c in chunks], "passage")

    for label, vecs, owners, texts in [
        ("документы", doc_vecs, [d.doc_id for d in docs], [d.text for d in docs]),
        ("чанки",     chunk_vecs, [c.doc_id for c in chunks], [c.text for c in chunks]),
    ]:
        top1 = top3 = total_chars = 0
        for question, expected in EVAL_QUESTIONS:
            q_vec = embed([question], "query")[0]
            order = np.argsort(vecs @ q_vec)[::-1]
            found_docs = []
            for i in order[:10]:
                if owners[i] not in found_docs:
                    found_docs.append(owners[i])
            top1 += found_docs[0] == expected
            top3 += expected in found_docs[:3]
            total_chars += len(texts[order[0]])
        print(f"e5 / {label:<10} top1={top1}/{len(EVAL_QUESTIONS)} "
              f"top3={top3}/{len(EVAL_QUESTIONS)} "
              f"средняя находка: {total_chars // len(EVAL_QUESTIONS)} симв.")

    # и главное - "Упал прод" лексика проваливала, а семантика?
    q_vec = embed(["Упал прод, что делать в первую очередь?"], "query")[0]
    best = chunks[int(np.argmax(chunk_vecs @ q_vec))]
    print(f"\n«Упал прод» нашёл: {best.chunk_id}")
    print(best.text[:180], "...")

Ожидаемая картина: семантика закрывает «Упал прод» (синонимы!), а чанки по-прежнему выигрывают в компактности находок. Оба улучшения сложились.

## Чек-лист чанкинга для продакшена

```text
1. Есть структура (абзацы, заголовки)? -> режьте по ней; окно - запасной путь
2. Размер: старт 300-600 символов, перекрытие 15-25% (для окна)
3. Потолок: лимит токенов модели эмбеддингов (e5-small: 512) - не превышать
4. Каждому чанку паспорт: doc_id, title, topic, chunk_index, source
5. Выдачу диверсифицировать по doc_id (но см. упражнение 3)
6. Любое изменение размера/стратегии -> прогон eval, а не "вроде лучше"
```

## ⚠️ Частые ошибки

1. **Чанк длиннее лимита эмбеддера.** Модель тихо обрежет хвост — хвост исчезнет из поиска навсегда, без единой ошибки в логах. Проверяйте длины при индексации.
2. **Резать по символам без отката до пробела/предложения.** Половина слова на границе — мусор в токенах и в векторе.
3. **Окно без перекрытия.** Мысль на границе чанков не найдётся ни по одному запросу.
4. **Чанки-сироты.** Забыли `doc_id`/`chunk_index` — не сможете ни показать источник, ни склеить контекст, ни дедуплицировать. Метаданные проектируются *до* индексации.
5. **Топ-3 из одного документа.** Без диверсификации многочанковый документ монополизирует выдачу.
6. **Реранкер поверх слабого recall.** Вторая ступень пересортировывает только то, что принесла первая. Сначала расширьте кандидатов, потом ужесточайте отбор.
7. **Подбор размера чанка «на глаз».** У вас уже есть eval — подбор размера это один цикл `for` (упражнение 1).

## 🏋️ Упражнения

### Упражнение 1 — размер решает 📏

Прогоните лексический eval для `chunk_paragraphs` с `max_chars` из {250, 500, 900} и для `chunk_fixed(400, 80)`. Сравните top-1/top-3 и средний размер находки. Какой конфиг выбрали бы для RAG, где каждый символ контекста стоит денег?

### Упражнение 2 — соседи по документу 🧩

Напишите `get_with_neighbors(chunk, chunks)`: вернуть текст чанка вместе с соседями (`chunk_index` ± 1 того же документа). Проверьте на любом чанке из середины документа. Это заготовка паттерна «нашли точным маленьким чанком — отдали LLM расширенный контекст» (в уроке 2.6 он вырастет в parent-document retrieval).

### Упражнение 3 — тёмная сторона диверсификации 🌓

Придумайте вопрос, на который *оба* лучших ответа лежат в **одном** документе (подсказка: в «Больничных» есть и правило трёх дней, и доплата до 100%). Покажите, что `rerank_diverse` с `max_per_doc=1` выкидывает второй по качеству фрагмент, и сравните с `max_per_doc=2`. Сформулируйте вывод о дефолтах.

## 📝 Мини-квиз

**1. Почему вектор целого документа проигрывает вектору чанка на конкретных вопросах?**

<details><summary>Ответ</summary>

Вектор документа — усреднение всех его подтем, точка «между» ними на карте смыслов. Вопрос пользователя попадает в конкретную подтему, и чанк этой подтемы оказывается ближе, чем «среднее». Чем длиннее и разнороднее документ, тем сильнее эффект.

</details>

**2. Зачем чанкам перекрытие и какое оно должно быть?**

<details><summary>Ответ</summary>

Мысль, разрезанная границей чанков, без перекрытия не найдётся ни по одному запросу — она есть только по половинке в каждом чанке. Перекрытие 15–25% размера дублирует границу в обоих чанках. Больше — раздувает индекс, меньше — не защищает.

</details>

**3. Что случится с чанком в 3000 символов при индексации моделью e5-small?**

<details><summary>Ответ</summary>

Лимит модели — 512 токенов (~1000–1500 символов русского): всё, что дальше, будет тихо отброшено при эмбеддинге. Хвост чанка не попадёт в вектор и станет ненаходимым — без ошибок и предупреждений. Тихая обрезка, как в уроке 1.5, только на индексации.

</details>

**4. Опишите двухступенчатый поиск и роль каждой ступени.**

<details><summary>Ответ</summary>

Этап 1 (recall): дешёвый поиск по всей базе — вектор/BM25, топ-20–50 кандидатов; важно ничего не упустить. Этап 2 (precision): дорогая пересортировка немногих кандидатов (cross-encoder, LLM, правила) — важно поставить лучшее наверх. Реранкер не вернёт то, что не принёс recall-этап.

</details>

**5. Почему «Упал прод» не нашёлся лексическим поиском ни по документам, ни по чанкам?**

<details><summary>Ответ</summary>

В документе про инциденты нет слов «упал» и «прод» — лексический поиск сравнивает буквы, а не смысл (bag-of-words из урока 2.1). Чанкинг меняет гранулярность, но не природу сравнения. Лечится эмбеддингами; чанкинг и семантика — независимые, складывающиеся улучшения.

</details>

## ✅ Решения упражнений

### Решение 1 — размер решает

In [ ]:
def eval_config(label: str, texts: list[str], owners: list[str]) -> None:
    token_lists, idf = build_lexical_index(texts)
    top1 = top3 = total_chars = 0
    for question, expected in EVAL_QUESTIONS:
        results = lexical_search(question, token_lists, idf, top_k=10)
        found_docs = []
        for _, i in results:
            if owners[i] not in found_docs:
                found_docs.append(owners[i])
        top1 += found_docs[0] == expected
        top3 += expected in found_docs[:3]
        total_chars += len(texts[results[0][1]])
    print(f"{label:<28} top1={top1}/{len(EVAL_QUESTIONS)}  top3={top3}/{len(EVAL_QUESTIONS)}  "
          f"находка ~{total_chars // len(EVAL_QUESTIONS)} симв.")

for max_chars in (250, 500, 900):
    cfg = chunk_documents(docs, max_chars=max_chars)
    eval_config(f"chunk_paragraphs({max_chars})",
                [c.text for c in cfg], [c.doc_id for c in cfg])

fixed_chunks = []
for doc in docs:
    for i, piece in enumerate(chunk_fixed(doc.text, chunk_size=400, overlap=80)):
        fixed_chunks.append((doc.doc_id, piece))
eval_config("chunk_fixed(400, 80)",
            [t for _, t in fixed_chunks], [d for d, _ in fixed_chunks])

В нашем прогоне таблица получилась говорящей: **оба конца шкалы наказаны**. Слишком мелкие чанки (250) уронили top-1 до 5/7 — фрагменты стали такими короткими, что растеряли сигнальные слова. Слишком крупные (900) держат точность, но находка распухла до целого документа — платить за неё в RAG-промпте придётся втрое. Золотая середина (500) сохранила 6/7 при находке в 370 символов. Это и есть «разрешение поиска» в числах: на своих данных всегда смотрите обе колонки — первую, чтобы не потерять качество, вторую — чтобы не платить за воздух.

### Решение 2 — соседи по документу

In [ ]:
def get_with_neighbors(chunk: Chunk, all_chunks: list[Chunk]) -> str:
    """Текст чанка + его соседи по документу (chunk_index +- 1), по порядку."""
    idx = chunk.metadata["chunk_index"]
    neighbors = [
        c for c in all_chunks
        if c.doc_id == chunk.doc_id
        and abs(c.metadata["chunk_index"] - idx) <= 1
    ]
    neighbors.sort(key=lambda c: c.metadata["chunk_index"])
    return "\n\n".join(c.text for c in neighbors)

# найдём точным вопросом маленький чанк - и развернём его в контекст
question = "Сколько компания доплачивает в первые дни болезни?"
_, i = lexical_search(question, chunk_tokens, chunk_idf, top_k=1)[0]
found = chunks[i]

print(f"Нашли: {found.chunk_id} ({len(found.text)} симв.)")
context = get_with_neighbors(found, chunks)
print(f"С соседями: {len(context)} симв.\n")
print(context[:400], "...")

Паттерн «ищем мелко, отдаём крупно»: маленький чанк даёт точное попадание, соседи возвращают контекст. Обе части работают на своих сильных сторонах — и всё это благодаря `chunk_index` в метаданных. В уроке 2.6 идея дорастёт до parent-document retrieval, где «соседями» служит весь родительский документ.

### Решение 3 — тёмная сторона диверсификации

In [ ]:
question = "Правило трёх дней доверия и доплата до ста процентов оклада"
candidates = lexical_search(question, chunk_tokens, chunk_idf, top_k=10)

print(f"Вопрос: {question}\n")
print("ЭТАП 1, топ-4 кандидата:")
for score, i in candidates[:4]:
    print(f"  {score:.3f}  {chunks[i].chunk_id}")

for max_per_doc in (1, 2):
    print(f"\nДиверсификация max_per_doc={max_per_doc}:")
    for score, i in rerank_diverse(candidates, chunks, max_per_doc=max_per_doc):
        print(f"  {score:.3f}  {chunks[i].chunk_id:<14} {chunks[i].text[:50]}...")

Вопрос celenaправленно бьёт в **два фрагмента одного документа** — и `max_per_doc=1` честно выкидывает второй по релевантности чанк ради «разнообразия», которое здесь никому не нужно. Вывод: диверсификация — это ручка под конкретный продукт, а не универсальное благо. Хорошие дефолты: `max_per_doc=2` и — как всегда — решение по eval'у, а не по красоте идеи.

## 🎓 Итоги

- Вектор документа — среднее его подтем; **чанк** — это разрешение поиска. Крайности плохи обе, рабочая зона — 300–600 символов с потолком в лимит токенов эмбеддера (тихая обрезка!).
- **Структура автора лучше слепого окна**: абзацы — готовые чанки; окно с перекрытием 15–25% — для текстов без структуры.
- **Метаданные — паспорт чанка**: doc_id, title, topic, chunk_index, source. Проектируются до индексации; без них фрагмент — сирота.
- Замер показал честно: на маленькой чистой базе чанкинг не меняет «какой документ нашёлся» — он в разы уменьшает и уточняет **то, что вы получаете**. А «Упал прод» показал: чанкинг не заменяет семантику — они складываются.
- **Двухступенчатый поиск**: дешёвый recall → дорогой precision. Диверсификация по doc_id — простейший реранкер; и даже у неё есть тёмная сторона.

**Дальше — урок 2.6 «Расширенные стратегии»:** semantic chunking (границы по смыслу, а не по символам), parent-document retrieval и настоящий cross-encoder-реранкер.

## 📚 Что почитать

- [Pinecone: Chunking Strategies](https://www.pinecone.io/learn/chunking-strategies/) — обзор стратегий с примерами
- [LangChain: Text Splitters](https://python.langchain.com/docs/concepts/text_splitters/) — как это устроено во фреймворках (мы написали то же руками)
- [Introduction to Information Retrieval, гл. 6](https://nlp.stanford.edu/IR-book/) — TF-IDF и скоринг: классика, стоящая за нашими 20 строками
- [Sentence-Transformers: Retrieve & Re-Rank](https://sbert.net/examples/applications/retrieve_rerank/README.html) — двухступенчатый поиск глазами авторов библиотеки (мостик к уроку 2.6)